In [1]:
!pip install pillow
!pip install -U transformers
!pip install datasets accelerate evaluate
!pip install segmentation-models-pytorch
!pip install albumentations --upgrade
!pip install huggingface_hub
!pip install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 71.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 33.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers-4.57.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.2.0 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.1 MB/s et

In [2]:
!pip install timm

In [3]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login via CLI
!hf auth login --token $hf_token

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `kaggle` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `kaggle`


In [4]:
# =========================
# Plant Disease Segmentation - DataLoader (on-the-fly)
# Resize + edge padding to 512x512, ImageNet normalize
# =========================

import os
from pathlib import Path
import numpy as np
import cv2
import torch
from torch.utils.data import Dataset, DataLoader

# ---- Config ----
DATA_ROOT = Path("/kaggle/input/plantseg-12k/PlantSeg05")  # <-- chỉnh nếu khác
IMG_SIZE = 512

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# ---- Helpers ----
def _list_images(folder: Path):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in exts])

def _pad_to_square_edge(img, target=512, is_mask=False):
    """
    img: HxWxC (uint8) or HxW (uint8/int)
    Pads to target x target using edge replication (copy border).
    """
    h, w = img.shape[:2]
    if h == target and w == target:
        return img

    # center padding
    top = (target - h) // 2
    bottom = target - h - top
    left = (target - w) // 2
    right = target - w - left

    # cv2 border types:
    # - For image: BORDER_REPLICATE is correct.
    # - For mask: BORDER_REPLICATE also fine to avoid introducing new class ids.
    border_type = cv2.BORDER_REPLICATE

    if img.ndim == 2:
        out = cv2.copyMakeBorder(img, top, bottom, left, right, border_type)
    else:
        out = cv2.copyMakeBorder(img, top, bottom, left, right, border_type)
    return out

def resize_with_edge_padding(img, target=512, interp=cv2.INTER_LINEAR, is_mask=False):
    """
    Keep aspect ratio: resize so the longer side == target, then edge-pad to target x target.
    - For image: interp linear
    - For mask: interp nearest (to preserve labels)
    """
    h, w = img.shape[:2]
    if h == 0 or w == 0:
        raise ValueError("Empty image encountered.")

    scale = target / max(h, w)
    new_h = int(round(h * scale))
    new_w = int(round(w * scale))

    # Ensure at least 1 pixel
    new_h = max(1, new_h)
    new_w = max(1, new_w)

    if (new_h, new_w) != (h, w):
        img = cv2.resize(img, (new_w, new_h), interpolation=interp)

    # Pad to target with edge replication
    img = _pad_to_square_edge(img, target=target, is_mask=is_mask)
    return img

def normalize_imagenet(img_rgb_float01):
    # img: float32, HxWx3 in [0,1]
    return (img_rgb_float01 - IMAGENET_MEAN) / IMAGENET_STD

# ---- Dataset ----
class PlantSegDataset(Dataset):
    def __init__(self, root: Path, split: str, target_size: int = 512):
        """
        root/split/images
        root/split/annotations
        """
        self.root = Path(root)
        self.split = split
        self.target_size = target_size

        self.img_dir = self.root / split / "images"
        self.ann_dir = self.root / split / "annotations"

        if not self.img_dir.exists():
            raise FileNotFoundError(f"Image dir not found: {self.img_dir}")
        if not self.ann_dir.exists():
            raise FileNotFoundError(f"Annotation dir not found: {self.ann_dir}")

        self.img_paths = _list_images(self.img_dir)
        if len(self.img_paths) == 0:
            raise RuntimeError(f"No images found in {self.img_dir}")

        # Build mapping from stem -> annotation path (any extension)
        ann_paths = _list_images(self.ann_dir)
        self.ann_map = {p.stem: p for p in ann_paths}

        # Filter only images that have masks
        kept = []
        missing = 0
        for p in self.img_paths:
            if p.stem in self.ann_map:
                kept.append(p)
            else:
                missing += 1
        self.img_paths = kept

        if len(self.img_paths) == 0:
            raise RuntimeError(f"No matching image-mask pairs in {self.img_dir} and {self.ann_dir}")
        if missing > 0:
            print(f"[Warning] {missing} images have no matching annotation (by stem) and were skipped.")

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        mask_path = self.ann_map[img_path.stem]

        # --- Read image (BGR uint8) -> RGB float32 ---
        img_bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise RuntimeError(f"Failed to read image: {img_path}")
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # --- Read mask (single channel) ---
        # If mask saved as RGB but with values, IMREAD_GRAYSCALE will still get channel 1.
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise RuntimeError(f"Failed to read mask: {mask_path}")

        # --- Resize with aspect ratio + edge padding ---
        img_rgb = resize_with_edge_padding(
            img_rgb, target=self.target_size, interp=cv2.INTER_LINEAR, is_mask=False
        )
        mask = resize_with_edge_padding(
            mask, target=self.target_size, interp=cv2.INTER_NEAREST, is_mask=True
        )

        # --- Normalize (ImageNet) ---
        img = img_rgb.astype(np.float32) / 255.0
        img = normalize_imagenet(img)

        # --- To tensor ---
        # image: [3, H, W], float32
        img_t = torch.from_numpy(img).permute(2, 0, 1).contiguous().float()
        # mask: [H, W], long (0..30)
        mask_t = torch.from_numpy(mask.astype(np.int64))

        sample = {
            "pixel_values": img_t,
            "mask": mask_t,
            "image_path": str(img_path),
            "mask_path": str(mask_path),
        }
        return sample

# ---- Dataloaders ----
def make_loader(split, batch_size=8, num_workers=2, shuffle=True):
    ds = PlantSegDataset(DATA_ROOT, split=split, target_size=IMG_SIZE)
    dl = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle if split == "train" else False,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=(split == "train"),
    )
    return ds, dl

train_ds, train_loader = make_loader("train", batch_size=4, num_workers=2, shuffle=True)
val_ds, val_loader     = make_loader("val",   batch_size=4, num_workers=2, shuffle=False)
test_ds, test_loader   = make_loader("test",  batch_size=4, num_workers=2, shuffle=False)

print("Train:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))

# ---- Quick sanity check ----
batch = next(iter(train_loader))
print(batch["pixel_values"].shape, batch["pixel_values"].dtype)  # [B,3,512,512] float32
print(batch["mask"].shape, batch["mask"].dtype, batch["mask"].min().item(), batch["mask"].max().item())


Train: 12000 Val: 963 Test: 1203
torch.Size([4, 3, 512, 512]) torch.float32
torch.Size([4, 512, 512]) torch.int64 0 23


In [5]:
# ============================================================
# DINOv3 ViT-S/16 backbone (timm) for 512x512
# - FIX dynamic_img_size error
# - Create model with img_size=512 so patch grid matches
# - Freeze base, train LoRA (+ optionally pos_embed if missing)
# - Attach hooks to capture block outputs for later FPN
# ============================================================

import torch
import torch.nn as nn
import timm

# ---------------------------
# 1) LoRA wrapper for Linear
# ---------------------------
class LoRALinear(nn.Module):
    """
    Wrap a nn.Linear with LoRA:
      y = W x + scale * (B(A(x)))

    - base W (pretrained) is frozen
    - A and B are trainable low-rank matrices
    """
    def __init__(self, base_linear: nn.Linear, r=8, alpha=16, dropout=0.0):
        super().__init__()
        assert isinstance(base_linear, nn.Linear)

        self.base = base_linear
        self.r = r
        self.alpha = alpha
        self.scale = alpha / r if r > 0 else 1.0
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        in_features = base_linear.in_features
        out_features = base_linear.out_features

        # LoRA params
        self.lora_A = nn.Parameter(torch.empty(r, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))

        # Init: A ~ kaiming, B = 0 (so LoRA starts as "no-op")
        nn.init.kaiming_uniform_(self.lora_A, a=5**0.5)
        nn.init.zeros_(self.lora_B)

        # Freeze base linear weights
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        out = self.base(x)
        x_d = self.dropout(x)
        lora = (x_d @ self.lora_A.t()) @ self.lora_B.t()
        return out + self.scale * lora


def _replace_module(parent: nn.Module, name: str, new_module: nn.Module):
    setattr(parent, name, new_module)


def inject_lora_into_vit(model: nn.Module, r=8, alpha=16, dropout=0.0,
                         target=("qkv", "proj", "fc1", "fc2")):
    """
    Inject LoRA into common timm ViT blocks:
      - blocks[i].attn.qkv
      - blocks[i].attn.proj
      - blocks[i].mlp.fc1
      - blocks[i].mlp.fc2
    """
    assert hasattr(model, "blocks"), "Not a timm ViT-like model (missing .blocks)"

    for block in model.blocks:
        # Attention
        if hasattr(block, "attn"):
            attn = block.attn
            if "qkv" in target and hasattr(attn, "qkv") and isinstance(attn.qkv, nn.Linear):
                _replace_module(attn, "qkv", LoRALinear(attn.qkv, r=r, alpha=alpha, dropout=dropout))
            if "proj" in target and hasattr(attn, "proj") and isinstance(attn.proj, nn.Linear):
                _replace_module(attn, "proj", LoRALinear(attn.proj, r=r, alpha=alpha, dropout=dropout))

        # MLP
        if hasattr(block, "mlp"):
            mlp = block.mlp
            if "fc1" in target and hasattr(mlp, "fc1") and isinstance(mlp.fc1, nn.Linear):
                _replace_module(mlp, "fc1", LoRALinear(mlp.fc1, r=r, alpha=alpha, dropout=dropout))
            if "fc2" in target and hasattr(mlp, "fc2") and isinstance(mlp.fc2, nn.Linear):
                _replace_module(mlp, "fc2", LoRALinear(mlp.fc2, r=r, alpha=alpha, dropout=dropout))

    return model


# ---------------------------
# 2) Hook collector
# ---------------------------
class BlockOutputHook:
    """
    Save token outputs from selected transformer blocks.
    Each hook stores output tokens of a block: (B, N, C)
    """
    def __init__(self, model: nn.Module, block_indices=(2, 5, 8, 11)):
        self.block_indices = list(block_indices)
        self.handles = []
        self.features = {}

        for idx in self.block_indices:
            h = model.blocks[idx].register_forward_hook(self._hook_fn(idx))
            self.handles.append(h)

    def _hook_fn(self, idx):
        def fn(module, inputs, output):
            self.features[idx] = output
        return fn

    def clear(self):
        self.features = {}

    def remove(self):
        for h in self.handles:
            h.remove()
        self.handles = []


# ---------------------------
# 3) Load checkpoint (flexible)
# ---------------------------
def load_checkpoint_flexible(model: nn.Module, ckpt_path: str, verbose=True):
    """
    Accepts:
    - OrderedDict state_dict (your dinov3 file is this)
    - dict with 'model' or 'state_dict'
    """
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    if isinstance(ckpt, dict) and not isinstance(ckpt, nn.Module):
        if "model" in ckpt and isinstance(ckpt["model"], dict):
            state = ckpt["model"]
        elif "state_dict" in ckpt and isinstance(ckpt["state_dict"], dict):
            state = ckpt["state_dict"]
        else:
            state = ckpt
    else:
        state = ckpt

    # strip "module." if any
    state = {k.replace("module.", ""): v for k, v in state.items()}

    msg = model.load_state_dict(state, strict=False)

    if verbose:
        print("Loaded ckpt with strict=False")
        print("  Missing keys:", len(msg.missing_keys))
        print("  Unexpected keys:", len(msg.unexpected_keys))
        if msg.missing_keys:
            print("  Missing examples:", msg.missing_keys[:5])
        if msg.unexpected_keys:
            print("  Unexpected examples:", msg.unexpected_keys[:5])

    return msg


# ---------------------------
# 4) Backbone wrapper (512x512)
# ---------------------------
class DINOv3ViTBackbone(nn.Module):
    """
    ViT-S/16 backbone for 512x512:
    - Build ViT with img_size=512 to avoid dynamic_img_size issues
    - Freeze all pretrained weights
    - Train only LoRA (+ pos_embed if ckpt doesn't provide it)
    - Hooks grab intermediate block outputs -> token maps -> feature maps
    """
    def __init__(
        self,
        ckpt_path: str,
        img_size: int = 512,
        vit_name: str = "vit_small_patch16_224",
        hook_blocks=(2, 5, 8, 11),
        lora_r: int = 8,
        lora_alpha: int = 16,
        lora_dropout: float = 0.0,
        lora_targets=("qkv", "proj", "fc1", "fc2"),
        train_pos_embed_if_missing: bool = True,
    ):
        super().__init__()

        # (A) Create model WITH img_size=512 so PatchEmbed grid + pos_embed length match 512.
        #     This avoids needing timm dynamic_img_size branch (which expects NHWC 4D tensors).
        try:
            self.vit = timm.create_model(
                vit_name,
                pretrained=False,
                num_classes=0,
                global_pool="",
                img_size=img_size,     # key fix
            )
        except TypeError:
            # Fallback if timm version doesn't accept img_size argument
            self.vit = timm.create_model(
                vit_name,
                pretrained=False,
                num_classes=0,
                global_pool="",
            )
            # enforce strict 512 only (since you always pad to 512 anyway)
            if hasattr(self.vit, "patch_embed"):
                pe = self.vit.patch_embed
                if hasattr(pe, "img_size"):
                    pe.img_size = (img_size, img_size)
                if hasattr(pe, "strict_img_size"):
                    pe.strict_img_size = True

        # IMPORTANT: ensure we do NOT enable dynamic_img_size (causes your error)
        if hasattr(self.vit, "dynamic_img_size"):
            self.vit.dynamic_img_size = False

        # (B) Load checkpoint
        msg = load_checkpoint_flexible(self.vit, ckpt_path, verbose=True)

        # (C) Freeze all base params
        for p in self.vit.parameters():
            p.requires_grad = False

        # (D) If checkpoint is missing pos_embed (your case), and the model uses pos_embed,
        #     pos_embed is random init => allow it to be trainable.
        if train_pos_embed_if_missing and hasattr(self.vit, "pos_embed") and self.vit.pos_embed is not None:
            if any("pos_embed" in k for k in msg.missing_keys):
                self.vit.pos_embed.requires_grad = True

        # (E) Inject LoRA (trainable)
        inject_lora_into_vit(
            self.vit,
            r=lora_r,
            alpha=lora_alpha,
            dropout=lora_dropout,
            target=lora_targets,
        )

        # (F) Attach hooks for multi-level features
        self.hook = BlockOutputHook(self.vit, block_indices=hook_blocks)
        self.hook_blocks = list(hook_blocks)

        # Patch size (ViT-S/16)
        ps = self.vit.patch_embed.patch_size
        self.patch_size = ps[0] if isinstance(ps, (tuple, list)) else int(ps)

    def _grid_hw(self, x):
        H, W = x.shape[-2], x.shape[-1]
        return H // self.patch_size, W // self.patch_size

    def forward(self, x):
        self.hook.clear()

        # forward triggers hooks
        _ = self.vit.forward_features(x)

        gh, gw = self._grid_hw(x)
        feats = {}
        for idx in self.hook_blocks:
            tok = self.hook.features[idx]   # (B, N, C)
            tok = tok[:, 1:, :]             # drop CLS -> (B, gh*gw, C)
            B, N, C = tok.shape
            tok = tok.reshape(B, gh, gw, C).permute(0, 3, 1, 2).contiguous()
            feats[f"block{idx}"] = tok
        return feats


# ---------------------------
# 5) Test with your dataloader
# ---------------------------

# Dùng đúng path weight của bạn trên Kaggle:
# - Nếu weight nằm trong /kaggle/input/... thì đặt path tương ứng
# - Nếu bạn upload file vào working dir thì path thường là "dinov3_....pth"
CKPT_PATH = "/kaggle/input/models/qunhanhvng/dinov3/pytorch/default/1/DINOv3/dinov3_vits16_pretrain_lvd1689m-08c60483.pth"

device = "cuda" if torch.cuda.is_available() else "cpu"

backbone = DINOv3ViTBackbone(
    ckpt_path=CKPT_PATH,
    img_size=512,
    vit_name="vit_small_patch16_224",
    hook_blocks=(2, 5, 8, 11),
    lora_r=8,
    lora_alpha=16,
    lora_dropout=0.0,
).to(device)

trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
total = sum(p.numel() for p in backbone.parameters())
print(f"Total params: {total/1e6:.2f}M | Trainable params: {trainable/1e6:.2f}M")

batch = next(iter(train_loader))
x = batch["pixel_values"].to(device)

feats = backbone(x)
for k, v in feats.items():
    print(k, v.shape, v.dtype)  # expected: (B, 384, 32, 32) for 512 with patch16


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Loaded ckpt with strict=False
  Missing keys: 1
  Unexpected keys: 39
  Missing examples: ['pos_embed']
  Unexpected examples: ['storage_tokens', 'mask_token', 'rope_embed.periods', 'blocks.0.attn.qkv.bias_mask', 'blocks.0.ls1.gamma']
Total params: 22.57M | Trainable params: 0.98M
block2 torch.Size([4, 384, 32, 32]) torch.float32
block5 torch.Size([4, 384, 32, 32]) torch.float32
block8 torch.Size([4, 384, 32, 32]) torch.float32
block11 torch.Size([4, 384, 32, 32]) torch.float32


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LayerNorm2d(nn.Module):
    """LayerNorm support for (B, C, H, W) layout"""
    def __init__(self, num_channels: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(num_channels))
        self.bias = nn.Parameter(torch.zeros(num_channels))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        x = self.weight[:, None, None] * x + self.bias[:, None, None]
        return x

class SimpleFeaturePyramid(nn.Module):
    """
    Biến đổi feature đơn tỷ lệ (Single-scale 1/16) từ ViT thành Pyramid (P2, P3, P4, P5).
    Đây là kỹ thuật chuẩn dùng trong ViTDet / Mask2Former với ViT backbone.
    """
    def __init__(
        self,
        in_channels=384,      # ViT-S/16 hidden dim
        out_channels=256,     # Mask2Former PixelDecoder dim (thường là 256)
        scale_factors=(4.0, 2.0, 1.0, 0.5), # Tương ứng P2(1/4), P3(1/8), P4(1/16), P5(1/32)
        target_key="block11"  # Key của feature muốn dùng làm gốc (thường dùng layer cuối)
    ):
        super().__init__()
        self.target_key = target_key
        self.stages = nn.ModuleList()

        for scale in scale_factors:
            # Logic tạo các tầng Pyramid từ input 1/16 (scale=1.0)
            # scale=4.0 => Upsample 4x (P2: 1/4 stride)
            # scale=2.0 => Upsample 2x (P3: 1/8 stride)
            # scale=1.0 => Identity    (P4: 1/16 stride)
            # scale=0.5 => Downsample 2x (P5: 1/32 stride)
            
            layers = []
            
            # 1. Projection ban đầu (384 -> 256)
            if scale == 1.0:
                layers.append(
                    nn.Sequential(
                        nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                        LayerNorm2d(out_channels),
                        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
                        LayerNorm2d(out_channels)
                    )
                )
            elif scale > 1.0: # Upsampling (P2, P3)
                # Dùng ConvTranspose2d để học upsampling tốt hơn interpolation
                scale_int = int(scale)
                layers.append(
                    nn.Sequential(
                        nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                        LayerNorm2d(out_channels),
                        nn.ConvTranspose2d(
                            out_channels, out_channels, 
                            kernel_size=2*scale_int, stride=scale_int, padding=scale_int//2
                        ),
                        LayerNorm2d(out_channels)
                    )
                )
            else: # Downsampling (P5)
                layers.append(
                    nn.Sequential(
                        nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                        LayerNorm2d(out_channels),
                        nn.MaxPool2d(kernel_size=2, stride=2), # Hoặc Conv stride 2
                        LayerNorm2d(out_channels)
                    )
                )
            
            self.stages.append(nn.Sequential(*layers))

    def forward(self, features_dict):
        """
        features_dict: Output từ DINOv3ViTBackbone 
                       {'block2': ..., 'block11': (B, 384, H/16, W/16)}
        """
        # Lấy feature gốc (thường là block cuối cùng)
        if self.target_key not in features_dict:
            raise KeyError(f"Key '{self.target_key}' not found in backbone output: {features_dict.keys()}")
        
        x = features_dict[self.target_key] # (B, 384, 32, 32)
        
        # Tạo pyramid
        outs = {}
        stage_names = ["p2", "p3", "p4", "p5"] # Mask2Former thường cần tên này hoặc list
        
        for stage, name in zip(self.stages, stage_names):
            outs[name] = stage(x)
            
        return outs

# ==========================================
# TEST KẾT HỢP (Backbone + FPN)
# ==========================================
if __name__ == "__main__":
    # 1. Giả lập Backbone (đã có từ code của bạn)
    # Lưu ý: Cần đảm bảo backbone đã được load weight
    # backbone = ... (instance bạn đã tạo ở trên)

    # 2. Khởi tạo FPN Adapter
    # Mask2Former thường yêu cầu features channel = 256
    fpn = SimpleFeaturePyramid(
        in_channels=384,   # ViT-S
        out_channels=256, 
        target_key="block11" # Dùng block sâu nhất để tạo pyramid
    ).to(device)

    # 3. Chạy thử
    print("-" * 30)
    print("Testing Pipeline...")
    # x đã có từ code cũ của bạn
    
    # Forward backbone
    backbone_feats = backbone(x)
    print(f"Backbone out keys: {backbone_feats.keys()}")
    
    # Forward FPN
    pyramid_feats = fpn(backbone_feats)
    
    # 4. Kiểm tra shape (Quan trọng cho Mask2Former)
    print("\nFeature Pyramid Outputs:")
    expected_strides = {"p2": 4, "p3": 8, "p4": 16, "p5": 32}
    for k, v in pyramid_feats.items():
        stride = 512 // v.shape[-1]
        print(f"{k.upper()}: Shape={v.shape} | Stride={stride} (Expected: {expected_strides[k]})")

------------------------------
Testing Pipeline...
Backbone out keys: dict_keys(['block2', 'block5', 'block8', 'block11'])

Feature Pyramid Outputs:
P2: Shape=torch.Size([4, 256, 128, 128]) | Stride=4 (Expected: 4)
P3: Shape=torch.Size([4, 256, 64, 64]) | Stride=8 (Expected: 8)
P4: Shape=torch.Size([4, 256, 32, 32]) | Stride=16 (Expected: 16)
P5: Shape=torch.Size([4, 256, 16, 16]) | Stride=32 (Expected: 32)


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SemanticFPNHead(nn.Module):
    """
    Nhận Feature Pyramid (P2, P3, P4, P5), gộp lại và dự đoán mask.
    Logic: Upsample tất cả về cùng size (P2: 1/4), nối lại (Concat) và Conv ra output.
    """
    def __init__(self, feature_channels=256, num_classes=30, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Sau khi concat 4 tầng (P2, P3, P4, P5) mỗi tầng 256 kênh -> tổng 1024 kênh
        self.fusion_conv = nn.Sequential(
            nn.Conv2d(feature_channels * 4, feature_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(feature_channels),
            nn.ReLU(inplace=True),
            self.dropout
        )
        
        # Layer cuối cùng dự đoán số class
        self.classifier = nn.Conv2d(feature_channels, num_classes, kernel_size=1)

    def forward(self, features_dict):
        # features_dict: {'p2': [B,256,128,128], 'p3': ..., 'p4': ..., 'p5': ...}
        
        p2 = features_dict['p2']
        p3 = features_dict['p3']
        p4 = features_dict['p4']
        p5 = features_dict['p5']
        
        # Target size là size của P2 (lớn nhất trong đám feature) = 128x128
        h, w = p2.shape[-2:]
        
        # Upsample các tầng thấp lên kích thước của P2
        p3_up = F.interpolate(p3, size=(h, w), mode='bilinear', align_corners=False)
        p4_up = F.interpolate(p4, size=(h, w), mode='bilinear', align_corners=False)
        p5_up = F.interpolate(p5, size=(h, w), mode='bilinear', align_corners=False)
        
        # Nối lại (Concatenate)
        x = torch.cat([p2, p3_up, p4_up, p5_up], dim=1) # [B, 1024, 128, 128]
        
        # Fusion & Predict
        x = self.fusion_conv(x)      # [B, 256, 128, 128]
        logits = self.classifier(x)  # [B, num_classes, 128, 128]
        
        return logits

class PlantDiseaseModel(nn.Module):
    def __init__(self, backbone, fpn, head, target_size=512):
        super().__init__()
        self.backbone = backbone
        self.fpn = fpn
        self.head = head
        self.target_size = target_size

    def forward(self, x):
        # 1. Backbone
        backbone_feats = self.backbone(x)
        
        # 2. Neck (FPN)
        pyramid_feats = self.fpn(backbone_feats)
        
        # 3. Head (Decoder) -> Trả về (B, num_classes, 128, 128)
        logits_small = self.head(pyramid_feats)
        
        # 4. Upsample cuối cùng ra 512x512
        logits = F.interpolate(
            logits_small, 
            size=(self.target_size, self.target_size), 
            mode='bilinear', 
            align_corners=False
        )
        return logits

In [8]:
# --- Config ---
NUM_CLASSES = 31 # Dataset của bạn có max index là 28, nên để dư ra (ví dụ 30) hoặc check kỹ len(classes)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Khởi tạo các thành phần (Đã có class từ cell trước)
# Lưu ý: Load lại backbone nếu cần để đảm bảo sạch sẽ
backbone = DINOv3ViTBackbone(
    ckpt_path=CKPT_PATH, 
    img_size=512, 
    lora_r=8
)

fpn = SimpleFeaturePyramid(
    in_channels=384, 
    out_channels=256, 
    target_key="block11"
)

head = SemanticFPNHead(
    feature_channels=256, 
    num_classes=NUM_CLASSES
)

# 2. Ghép thành Model tổng
model = PlantDiseaseModel(backbone, fpn, head).to(DEVICE)

# 3. Setup Optimizer: CHỈ TRAIN params có requires_grad=True
# (Tức là: LoRA A/B, PosEmbed (nếu có), FPN weights, Head weights)
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-4)

# Check lại số lượng tham số sẽ train
total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in trainable_params)
print(f"Model ready! Trainable: {train_p/1e6:.2f}M / Total: {total_p/1e6:.2f}M")

Loaded ckpt with strict=False
  Missing keys: 1
  Unexpected keys: 39
  Missing examples: ['pos_embed']
  Unexpected examples: ['storage_tokens', 'mask_token', 'rope_embed.periods', 'blocks.0.attn.qkv.bias_mask', 'blocks.0.ls1.gamma']
Model ready! Trainable: 9.58M / Total: 31.17M


In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

# ==========================================
# 1. Class tính toán Metrics (Bản Tối Ưu)
# ==========================================
class ConfusionMatrix:
    def __init__(self, num_classes):
        self.num_classes = num_classes
        self.reset()

    def reset(self):
        self.mat = np.zeros((self.num_classes, self.num_classes), dtype=np.int64)

    def update(self, a, b):
        """
        a: predicted masks (flattened)
        b: target masks (flattened)
        """
        # Loại bỏ các giá trị ignore (ví dụ 255) nếu có
        # Giả sử mask hợp lệ nằm trong [0, num_classes-1]
        k = (b >= 0) & (b < self.num_classes)
        a = a[k]
        b = b[k]
        
        # Tính bincount để cập nhật ma trận
        n = self.num_classes
        inds = n * b + a
        self.mat += np.bincount(inds, minlength=n**2).reshape(n, n)

    def compute(self):
        h = self.mat
        
        # --- 1. Pixel Accuracy (Global) ---
        acc_global = np.diag(h).sum() / (h.sum() + 1e-10)
        
        # --- 2. Class Accuracy (mAcc) ---
        # Acc = TP / (TP + FN) (Recall)
        # Chỉ tính trên các class CÓ xuất hiện trong target (h.sum(1) > 0)
        class_tp = np.diag(h)
        class_total = h.sum(axis=1)
        
        # Mask các class có xuất hiện
        mask = class_total > 0
        acc_cls = np.zeros(self.num_classes)
        acc_cls[mask] = class_tp[mask] / class_total[mask]
        mAcc = acc_cls[mask].mean() if mask.sum() > 0 else 0.0
        
        # --- 3. Mean IoU (mIoU) ---
        # IoU = TP / (TP + FP + FN)
        union = class_total + h.sum(axis=0) - class_tp
        
        # Chỉ tính IoU trên các class có Union > 0 (tránh chia cho 0)
        mask_iou = union > 0
        iou_cls = np.zeros(self.num_classes)
        iou_cls[mask_iou] = class_tp[mask_iou] / union[mask_iou]
        mIoU = iou_cls[mask_iou].mean() if mask_iou.sum() > 0 else 0.0
        
        return {
            "mIoU": mIoU,
            "mAcc": mAcc,
            "val_loss": 0.0 # Placeholder
        }

# ==========================================
# 2. Full Training Function
# ==========================================
def train_segmentation(
    model, 
    train_loader, 
    val_loader, 
    optimizer, 
    criterion, 
    num_epochs=20, 
    device="cuda", 
    save_dir="output_models"
):
    os.makedirs(save_dir, exist_ok=True)
    best_miou = 0.0
    
    # Sử dụng mixed precision
    scaler = torch.amp.GradScaler('cuda')
    
    # Metric tracker
    metric_tracker = ConfusionMatrix(num_classes=NUM_CLASSES)

    for epoch in range(num_epochs):
        print(f"\n🚀 Epoch {epoch+1}/{num_epochs}")
        
        # --- TRAIN ---
        model.train()
        train_loss = 0
        pbar = tqdm(train_loader, desc="Training")
        
        for batch in pbar:
            images = batch["pixel_values"].to(device)
            masks = batch["mask"].to(device)
            
            optimizer.zero_grad()
            
            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss = criterion(logits, masks)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
        avg_train_loss = train_loss / len(train_loader)
        
        # --- VALIDATE ---
        model.eval()
        val_loss = 0
        metric_tracker.reset()
        
        print("Validating...")
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Val"):
                images = batch["pixel_values"].to(device)
                masks = batch["mask"].to(device)
                
                with torch.amp.autocast('cuda'):
                    logits = model(images)
                    loss = criterion(logits, masks)
                
                val_loss += loss.item()
                
                # Update Metrics
                preds = torch.argmax(logits, dim=1)
                preds_np = preds.cpu().numpy().flatten()
                masks_np = masks.cpu().numpy().flatten()
                metric_tracker.update(preds_np, masks_np)

        avg_val_loss = val_loss / len(val_loader)
        
        # Tính toán kết quả
        metrics = metric_tracker.compute()
        current_miou = metrics["mIoU"]
        current_macc = metrics["mAcc"]
        
        # In kết quả ra màn hình (Đây là phần bạn cần)
        print(f"📊 Summary Epoch {epoch+1}:")
        print(f"   Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(f"   mIoU: {current_miou:.4f} ({current_miou*100:.2f}%)")
        print(f"   mAcc: {current_macc:.4f} ({current_macc*100:.2f}%)")
        
        # --- SAVE BEST MODEL ---
        if current_miou > best_miou:
            best_miou = current_miou
            save_path = os.path.join(save_dir, "best_model.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'miou': best_miou,
            }, save_path)
            print(f"New Best MIoU! Saved model to {save_path}")
        
        # Lưu model cuối cùng
        last_path = os.path.join(save_dir, "last_model.pth")
        torch.save(model.state_dict(), last_path)

    print(f"\n✅ Training Complete. Best MIoU: {best_miou:.4f}")

# ==========================================
# 3. RUN TRAINING
# ==========================================
# Chạy train
criterion = nn.CrossEntropyLoss()
train_segmentation(
    model, 
    train_loader, 
    val_loader, 
    optimizer, 
    criterion, 
    num_epochs=50, 
    device=DEVICE,
    save_dir="output_models"
)


🚀 Epoch 1/20


Training: 100%|██████████| 3000/3000 [12:04<00:00,  4.14it/s, loss=0.5099]  


Validating...


Val: 100%|██████████| 241/241 [00:43<00:00,  5.59it/s]


📊 Summary Epoch 1:
   Train Loss: 1.0987 | Val Loss: 1.0028
   mIoU: 0.0460 (4.60%)
   mAcc: 0.0663 (6.63%)
New Best MIoU! Saved model to output_models/best_model.pth

🚀 Epoch 2/20


Training:  49%|████▊     | 1461/3000 [05:21<05:36,  4.57it/s, loss=0.9648]

In [ ]:
# import torch
# import cv2
# import numpy as np
# import matplotlib.pyplot as plt
# from torchvision import transforms

# # ==========================================
# # 1. Hàm dự đoán (Đã cập nhật preprocess của bạn)
# # ==========================================
# def predict_image(image_path, model, device):
#     # --- B1: Đọc ảnh ---
#     img_bgr = cv2.imread(image_path)
#     if img_bgr is None:
#         print(f"Không đọc được ảnh: {image_path}")
#         return None, None

#     img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
#     # --- B2: Resize & Padding (Dùng hàm của bạn) ---
#     # Việc này giữ nguyên tỉ lệ ảnh, tránh làm méo các đốm bệnh
#     img_resized = resize_with_edge_padding(
#         img_rgb, 
#         target=512, 
#         interp=cv2.INTER_LINEAR, 
#         is_mask=False
#     )
    
#     # --- B3: Normalize (Dùng hàm của bạn) ---
#     # Chuyển sang float32 [0, 1] trước khi normalize
#     img_float = img_resized.astype(np.float32) / 255.0
#     img_norm = normalize_imagenet(img_float)
    
#     # --- B4: To Tensor ---
#     # [H, W, 3] -> [3, H, W] -> [1, 3, H, W]
#     img_tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).float().to(device)
    
#     # --- B5: Predict ---
#     model.eval()
#     with torch.no_grad():
#         logits = model(img_tensor)
#         # Lấy class có xác suất cao nhất tại mỗi pixel
#         pred_mask = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy() # Kết quả: [512, 512]
        
#     return img_resized, pred_mask

# # ==========================================
# # 2. Setup Model & Load Weights (Chạy lại nếu cần)
# # ==========================================
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# NUM_CLASSES = 30 # Đảm bảo khớp với lúc train

# # Nếu bạn chưa khởi tạo model trong session này thì chạy lại đoạn này:
# # backbone = DINOv3ViTBackbone(ckpt_path=CKPT_PATH, img_size=512, lora_r=8)
# # fpn = SimpleFeaturePyramid(in_channels=384, out_channels=256, target_key="block11")
# # head = SemanticFPNHead(feature_channels=256, num_classes=NUM_CLASSES)
# # model = PlantDiseaseModel(backbone, fpn, head)

# # Load best weights
# BEST_MODEL_PATH = "/kaggle/working/output_models/best_model.pth"
# if os.path.exists(BEST_MODEL_PATH):
#     print(f"Loading weights from {BEST_MODEL_PATH}...")
#     checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=False)
#     model.load_state_dict(checkpoint['model_state_dict'])
#     model.to(DEVICE)
#     print("Model loaded successfully!")
# else:
#     print("⚠️ Chưa tìm thấy file weights. Hãy train xong trước nhé.")

# # ==========================================
# # 3. Chạy thử trên 1 ảnh Test
# # ==========================================
# # Lấy ngẫu nhiên 1 ảnh từ tập Test Loader để kiểm tra
# if len(test_ds) > 0:
#     # Lấy đường dẫn ảnh đầu tiên trong tập test
#     test_img_path = str(test_ds.img_paths[0])
    
#     # Hoặc bạn có thể thay bằng đường dẫn ảnh bên ngoài:
#     # test_img_path = "/kaggle/input/test_image.jpg"

#     print(f"Testing on: {test_img_path}")
#     original, mask = predict_image(test_img_path, model, DEVICE)

#     if original is not None:
#         plt.figure(figsize=(12, 6))
        
#         # Ảnh gốc (Đã qua resize + padding)
#         plt.subplot(1, 2, 1)
#         plt.imshow(original)
#         plt.title("Preprocessed Image (512x512)")
#         plt.axis('off')
        
#         # Mask dự đoán
#         plt.subplot(1, 2, 2)
#         # Dùng cmap='nipy_spectral' hoặc 'jet' để phân biệt rõ các class
#         plt.imshow(mask, cmap='nipy_spectral', interpolation='nearest') 
#         plt.title("Predicted Mask")
#         plt.axis('off')
        
#         plt.tight_layout()
#         plt.show()
        
#         print("Classes detected:", np.unique(mask))

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# # Giả sử 'mask' là kết quả dự đoán (numpy array 512x512) bạn vừa có ở trên
# unique, counts = np.unique(mask, return_counts=True)
# pixel_stats = dict(zip(unique, counts))

# print("📊 Thống kê số lượng pixel từng class:")
# for class_id, count in pixel_stats.items():
#     percentage = (count / (512*512)) * 100
#     print(f" - Class {class_id}: {count} pixels ({percentage:.2f}%)")

# # --- Vẽ tách riêng từng class để soi ---
# fig, axes = plt.subplots(1, len(unique), figsize=(15, 5))
# if len(unique) == 1: axes = [axes] # Handle trường hợp chỉ có 1 class

# for ax, class_id in zip(axes, unique):
#     # Tạo mask nhị phân cho riêng class đó
#     binary_mask = (mask == class_id).astype(np.uint8)
    
#     ax.imshow(binary_mask, cmap='gray')
#     ax.set_title(f"Class {class_id} Only")
#     ax.axis('off')

# plt.show()